In [ ]:
!pwd

In [ ]:
!ls

In [ ]:
import numpy as np
import matplotlib.pylab as plt

# Specify the paths to the files
stim_file_path = './output_020424_ds1/stimmed.txt'
C_file_path = './output_020424_ds1/analysis_proc_C.txt'
photo_file_path = './output_020424_ds1/photostimmed_msgs.txt'

# Load the files
stim = np.loadtxt(stim_file_path)
C = np.loadtxt(C_file_path)
photostim = np.loadtxt(photo_file_path)

In [ ]:
photostim.shape
#each entry of the vector 

In [ ]:
stim.shape

In [ ]:
C.shape

In [ ]:
plt.imshow(C)

In [ ]:
stim[:,2]

In [ ]:
stim_red = stim[:,[0,2]]

In [ ]:
np.unique(stim_red[:,1]).shape

In [ ]:
duration = 20
init_dur = 0
fig, ax = plt.subplots(figsize=(12,6))
t1, t2 = (0,500)
plt.plot(C[65,:t2])
for i in np.arange(stim.shape[0]):
    if stim[i,0]+duration <= t2: # and stim[i,0]-2 >= t1:
        ax.axvspan(stim[i,0]+init_dur, stim[i,0]+duration, alpha=0.2, color='gray')
plt.show()

In [ ]:
num_angles = 24
tc = np.zeros((C.shape[0], num_angles))
tc_var = np.zeros_like(tc)
avg_resp = np.zeros((C.shape[0], duration-init_dur, num_angles))
stim_cnt = np.zeros((num_angles))

angle = np.unique(stim[:,2])

duration = 20
init_dur = 2
bkgrd_dur = 5

In [ ]:
for i,s in enumerate(stim):
    if int(s[0])<C.shape[1]:
        a_ind = np.squeeze(np.argwhere(angle==s[2]))
        # import pdb; pdb.set_trace()

        # f_ind = np.squeeze(np.argwhere(freq==s[4]))
        bkgrnd = np.mean(C[:,int(s[0])-bkgrd_dur:int(s[0])+init_dur], axis=1)
        # tc[:, a_ind, v_ind, f_ind] = (np.sum(C[:,int(s[0])+init_dur:int(s[0])+duration], axis=1) - bkgrnd)
        # tc_var[:, a_ind, v_ind, f_ind] = np.std(C[:,int(s[0])+init_dur:int(s[0])+duration], axis=1)
        tc[:, a_ind] += (np.sum(C[:,int(s[0])+init_dur:int(s[0])+duration]-bkgrnd[...,np.newaxis], axis=1))
        if C[:,int(s[0])+init_dur:int(s[0])+duration].shape[1] == duration:
            avg_resp[:, :, a_ind] += C[:,int(s[0])+init_dur:int(s[0])+duration]

        stim_cnt[a_ind] += 1

tc = tc / stim_cnt / duration
avg_resp = avg_resp / stim_cnt / duration

In [ ]:
print(tc.shape)

In [ ]:
plt.figure(figsize=(15,15))
plt.imshow(tc.T)

In [ ]:
plt.hist(np.max(tc, axis=1))
np.count_nonzero(np.max(tc, axis=1)>250)

In [ ]:
fig = plt.figure(figsize=(15,5))
ax = plt.gca()
# ax = fig.add_subplot(projection='3d')
# color = ['limegreen', 'yellow', 'darkorange', 'orangered', 'purple', 'indigo', 'mediumblue', 'dodgerblue', 'cyan', 'limegreen']
cmap = plt.cm.get_cmap('rainbow', 24)
color_vel = cmap(range(24))

plt.plot(np.mean(C[:,:1000], axis=0), 'k--')
plt.plot(C[65,:1000], 'k')

for i in range(stim.shape[0]):
    # c = int(stim[i][2]/45)
    c = int(stim[i][2]/15)
    # c = int(np.argwhere(vel == stim[i][3]))
    if stim[i][0]+10 <= 1000:
        try:
            ax.axvspan(stim[i][0], stim[i][0]+20, color=color_vel[c], alpha=0.3)
        except:
            pass
plt.show()

In [ ]:
plt.polar(angle*(2*np.pi)/360, tc[65])
plt.gca().set_theta_zero_location("N")
plt.gca().set_theta_direction("clockwise")

In [ ]:
angle

In [ ]:
tc[65]

In [ ]:
plt.plot(tc[65])

In [ ]:
plt.figure(figsize=(20,20))

ii = 0
for n in range(C.shape[0]):
    if np.max(tc[n]) > 250:
        ax = plt.subplot(20, 8, ii+1, projection='polar')
        ax.plot(angle*(2*np.pi)/360, tc[n])
        ax.axis('off')
        plt.gca().set_theta_zero_location("N")
        plt.gca().set_theta_direction("clockwise")
#         ax.set_title(str(n))
        ii += 1
plt.tight_layout()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
def z_score(X):
    # X: ndarray, shape (n_features, n_samples)
    ss = StandardScaler(with_mean=True, with_std=True)
    Xz = ss.fit_transform(X.T).T
    return Xz

X = C#[300:, 5000:-1]

Xz = z_score(X)
c = np.cov(Xz, rowvar=True) # covariance matrix
eig_vals, eig_vecs = np.linalg.eig(c)
srt = np.argsort(eig_vals)[::-1]
# print('Sorted eigenvalues: \n{}'.format(eig_vals[srt]))
# print('\nFirst eigenvector: \n{}'.format(eig_vecs[:, srt][:, 0]))

plt.plot(eig_vals[srt][:50])

In [ ]:
N = 5
pca = PCA(n_components=N, whiten=False, random_state=1337)
X_pca = pca.fit_transform(Xz.T).T
print(X_pca.shape)

fig, ax = plt.subplots()

ax.plot(X_pca[0,:], X_pca[1,:], '.')
ax.plot(X_pca[0,100:120], X_pca[1,100:120], 'k-')
# add color

ax.set_xlabel('PC0')
ax.set_ylabel('PC1')

In [ ]:
color = ['limegreen', 'yellow', 'darkorange', 'orangered', 'purple', 'indigo', 'mediumblue', 'dodgerblue', 'cyan']


In [ ]:
fig, ax = plt.subplots()

for i in range(stim.shape[0]):
    c = int(stim[i][2]/45)
    t = int(stim[i][0])#-5000
    ax.plot(X_pca[4,t:t+10], X_pca[1,t:t+10], '.', color=color[c])
# add color

ax.set_xlabel('PC0')
ax.set_ylabel('PC1')

In [ ]:
Xz = z_score(tc.T)
c = np.cov(Xz, rowvar=True) # covariance matrix
eig_vals, eig_vecs = np.linalg.eig(c)
srt = np.argsort(eig_vals)[::-1]
# print('Sorted eigenvalues: \n{}'.format(eig_vals[srt]))
# print('\nFirst eigenvector: \n{}'.format(eig_vecs[:, srt][:, 0]))

plt.plot(eig_vals[srt])

In [ ]:
N = 5
pca = PCA(n_components=N, whiten=False, random_state=1337)
X_pca = pca.fit_transform(Xz.T).T
print(X_pca.shape)

fig, ax = plt.subplots()

ax.plot(X_pca[0,:], X_pca[1,:], '.')
# ax.plot(X_pca[0,100:120], X_pca[1,100:120], 'k-')
# add color

ax.set_xlabel('PC0')
ax.set_ylabel('PC1')

In [ ]:
fig, ax = plt.subplots()
from math import floor

for i in range(tc.shape[0]):
    c = floor(np.argmax(tc[i])/3)
    ax.plot(X_pca[4,i], X_pca[1,i], '.', color=color[c])
# add color

ax.set_xlabel('PC0')
ax.set_ylabel('PC1')